# Домашнее задание: Агентная система анализа документов
### Цель: Создать полностью функциональную систему, объединяющую:

* SmolAgents - создание умных агентов с инструментами
* LlamaIndex - индексирование и поиск документов
* LangGraph - оркестрация процессов в граф-конвейер




## Система оценки (всего 10 баллов)

| Часть | Задание | Баллы |
|-------|---------|-------|
| **ЧАСТЬ 1** | SmolAgents: Создать 2 новых Tool'а | **3 балла** |
| **ЧАСТЬ 2** | LlamaIndex: Создать retriever | **3 балла** |
| **ЧАСТЬ 3** | LangGraph: Доработать граф | **3 балла** |
| **ЧАСТЬ 4** | Документация и отчет | **1 балл** |
| | **ИТОГО** | **10 баллов** |



In [1]:
!pip install openai -q
!pip install smolagents -q
!pip install llama-index -q
!pip install langgraph -q
!pip install llama-index-embeddings-huggingface -q

In [2]:
!pip install ddgs

In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.schema import QueryBundle

from openai import OpenAI
from smolagents import tool

import os

print("Импорты готовы")


Импорты готовы


/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Конфигурируем OpenRouter
OPENROUTER_API_KEY = ""  # TODO: Вставь свой ключ

# Инициализируем OpenRouter как OpenAI-совместимый клиент
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# Обертка для упрощённого использования
class OpenRouterLLM:
    def __init__(self, api_key: str, model: str = "qwen/qwen-2.5-72b-instruct"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1"
        )
        self.model = model

    def generate(self, prompt: str, max_tokens: int = 500) -> str:
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Ошибка: {str(e)}"

    def __call__(self, prompt: str) -> str:
        return self.generate(prompt)

# Инициализируем LLM
llm = OpenRouterLLM(api_key=OPENROUTER_API_KEY)

print("OpenRouter конфигурирован")


OpenRouter конфигурирован


### ЧАСТЬ 1: SmolAgents - Создание инструментов
### Цель: Научиться создавать инструменты (Tools) для агентов.



#### Что вам нужно сделать: Добавить 2 новых Tool'а помимо готовых. (3 балла)

In [5]:
# Определяем Tools

@tool
def ask_openrouter(question: str) -> str:
    """
    Задаёт вопрос OpenRouter и получает ответ.

    Args:
        question: Вопрос для отправки в OpenRouter

    Returns:
        Ответ от модели OpenRouter
    """
    return llm(question)

# TO DO: реализуйте tool, который возвращает погоду в Москве и какой-то ещё на свой вкус (главное, чтобы внутри была какая-то логика с использованием внешних инструментов)

# Проверка:
# your_tool_name(...)  # Вызовите ваш tool для проверки

from smolagents import tool
import requests
import random

# --- Tool 1: Погода в Москве ---
@tool
def get_moscow_weather() -> str:
    """
    Возвращает текущую погоду в Москве.
    Использует внешний API для получения данных (например, open-meteo).
    """
    try:
        # Пример запроса к бесплатному API погоды
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": 55.7558,
                "longitude": 37.6176,
                "current_weather": True
            }
        ).json()

        temp = response["current_weather"]["temperature"]
        wind = response["current_weather"]["windspeed"]
        return f"Сейчас в Москве {temp}°C, скорость ветра {wind} км/ч."
    except Exception as e:
        return f"Не удалось получить погоду: {str(e)}"


# --- Tool 2: Генератор случайной шутки ---
@tool
def random_joke() -> str:
    """
    Возвращает случайную шутку из заранее заданного списка.
    """
    jokes = [
        "Почему программисты путают Хэллоуин и Рождество? Потому что Oct 31 = Dec 25!",
        "Как назвать программиста, который любит гулять? Scriptwalker.",
        "Почему компьютер всегда голоден? Потому что он ест байты."
    ]
    return random.choice(jokes)


# --- Проверка работы ---
print(get_moscow_weather())
print(random_joke())



Сейчас в Москве 1.9°C, скорость ветра 6.9 км/ч.
Почему компьютер всегда голоден? Потому что он ест байты.


In [6]:
from smolagents import Model, ChatMessage
from typing import List, Optional, Dict, Any

class CustomRouterModel(Model):
    def __init__(self, router_llm: OpenRouterLLM, **model_kwargs):
        super().__init__(**model_kwargs)
        self.router_llm = router_llm

    def generate(
        self,
        messages: List[Any],  # может быть List[dict] или List[ChatMessage]
        stop_sequences: Optional[List[str]] = None,
        response_format: Optional[Dict[str, str]] = None,
        tools_to_call_from: Optional[List[Any]] = None,
        **kwargs
    ) -> ChatMessage:
        # Подготовка prompt-а из сообщений
        parts = []
        for m in messages:
            if isinstance(m, ChatMessage):
                role = m.role
                content = m.content
            elif isinstance(m, dict):
                role = m.get("role", "user")
                content = m.get("content", "")
            else:
                # вдруг другой формат
                role = "user"
                content = str(m)

            parts.append(f"{role}: {content}")

        prompt = "\n".join(parts)

        # Вызов OpenRouter LLM
        text = self.router_llm.generate(prompt, **kwargs)

        # Возвращаем ChatMessage
        return ChatMessage(role="assistant", content=text)


In [7]:
from smolagents import ToolCallingAgent, tool

# Наши инструменты:
# ask_openrouter, get_moscow_weather, random_joke

tools = [ask_openrouter, get_moscow_weather, random_joke]
#custom_model = CustomRouterModel(llm)
custom_model = CustomRouterModel(llm, temperature=0.7, max_tokens=500)
# Создаём агента
agent = ToolCallingAgent(
    name="DocHelperAgent",
    description="Агент для работы с документами и получения информации",
    tools=tools,
    model=custom_model
)


In [8]:
# Запускаем агента
result = agent.run("Какая сейчас погода в Москве?")
print(result)

╭─────────────────────────────────────────── New run - DocHelperAgent ────────────────────────────────────────────╮
│                                                                                                                 │
│ Какая сейчас погода в Москве?                                                                                   │
│                                                                                                                 │
╰─ CustomRouterModel - None ──────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_moscow_weather' with arguments: {}                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Сейчас в Москве 1.9°C, скорость ветра 6.9 км/ч.

[Step 1: Duration 1.36 seconds]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Сейчас в Москве 1.9°C, скорость ветра 6.9 км/ч.'}      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Сейчас в Москве 1.9°C, скорость ветра 6.9 км/ч.

Final answer: Сейчас в Москве 1.9°C, скорость ветра 6.9 км/ч.

[Step 2: Duration 1.42 seconds]

Сейчас в Москве 1.9°C, скорость ветра 6.9 км/ч.


In [9]:
result = agent.run("Расскажи анегдот")
print(result)

╭─────────────────────────────────────────── New run - DocHelperAgent ────────────────────────────────────────────╮
│                                                                                                                 │
│ Расскажи анегдот                                                                                                │
│                                                                                                                 │
╰─ CustomRouterModel - None ──────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'random_joke' with arguments: {}                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Как назвать программиста, который любит гулять? Scriptwalker.

[Step 1: Duration 1.11 seconds]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Как назвать программиста, который любит гулять?        │
│ Scriptwalker.'}                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Как назвать программиста, который любит гулять? Scriptwalker.

Final answer: Как назвать программиста, который любит гулять? Scriptwalker.

[Step 2: Duration 3.24 seconds]

Как назвать программиста, который любит гулять? Scriptwalker.


## LlamaIndex - Работа с документами
### Цель: Научиться загружать, индексировать и искать документы.

### Что вам нужно сделать: создать retriever. (3 балла)
* Добавьте больше документов по разным тематикам, ML, DL, NLP, Agents
* Реализуйте ретривер с локальными эмбеддингами на llama_index

In [10]:
# Создаём тестовые документы
os.makedirs("./test_documents", exist_ok=True)

documents_content = {
    "ai_basics.txt": """
    ИСКУССТВЕННЫЙ ИНТЕЛЛЕКТ: ОСНОВЫ

    Искусственный интеллект (AI) — это технология, которая позволяет компьютерам
    выполнять задачи, обычно требующие человеческого интеллекта.

    Основные направления:
    - Машинное обучение: обучение на данных
    - Глубокое обучение: нейронные сети
    - Обработка естественного языка: работа с текстом
    - Компьютерное зрение: анализ изображений
    """,

    "ml_guide.txt": """
    МАШИННОЕ ОБУЧЕНИЕ: ПОЛНОЕ РУКОВОДСТВО

    Машинное обучение включает три основных типа:

    1. Обучение с учителем (Supervised Learning)
       - Используются размеченные данные
       - Примеры: SVM, Random Forest, нейросети

    2. Обучение без учителя (Unsupervised Learning)
       - Работает с неразмеченными данными
       - Примеры: K-means, иерархическая кластеризация

    3. Обучение с подкреплением (Reinforcement Learning)
       - Агент учится взаимодействовать с окружением
       - Примеры: игры, робототехника
    """,

    "nlp_basics.txt": """
    ОБРАБОТКА ЕСТЕСТВЕННОГО ЯЗЫКА (NLP)

    Natural Language Processing (NLP) — это область AI, которая работает с текстом.

    Основные задачи NLP:
    1. Анализ настроений - определение эмоций в тексте
    2. Машинный перевод - перевод между языками
    3. Извлечение информации - выделение фактов из текста
    4. Генерация текста - создание нового содержимого
    5. Вопросно-ответная система - ответы на вопросы
    """
}

# Сохраняем документы
for filename, content in documents_content.items():
    with open(f"./test_documents/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print(f"Созданы {len(documents_content)} тестовых документа")


Созданы 3 тестовых документа


In [11]:
import os

os.makedirs("./test_documents", exist_ok=True)

# Темы и шаблоны текстов
topics = {
    "ml_supervised_learning": "Машинное обучение — это процесс, при котором алгоритм обучается на размеченных данных...",
    "ml_unsupervised_learning": "Обучение без учителя — метод, где данные не имеют меток...",
    "ml_reinforcement_learning": "Обучение с подкреплением (Reinforcement Learning) — агент учится взаимодействовать с окружением...",
    "ml_ensembles": "Ансамблевые методы объединяют несколько моделей для повышения точности...",
    "ml_generative_models": "Генеративные модели создают новые данные, похожие на обучающие примеры...",
    "ml_bayesian": "Байесовские методы в ML основаны на теореме Байеса и вероятностных предположениях...",
    "ml_regression_classification": "Регрессия и классификация — два фундаментальных типа задач в ML...",
    "ml_decision_trees": "Деревья решений — простой, но мощный алгоритм классификации и регрессии...",
    "ml_svm": "Метод опорных векторов (SVM) применяется для разделения данных с максимальным зазором...",
    "ml_boosting": "Градиентный бустинг — ансамблевый метод, который строит модели последовательно...",
    "ml_feature_engineering": "Feature Engineering — это создание признаков из сырых данных...",
    "ml_cross_validation": "Кросс-валидация используется для оценки производительности модели...",
    "ml_missing_data": "Работа с пропущенными данными — важная часть препроцессинга...",
    "ml_class_imbalance": "Балансировка классов решает проблему, когда одни классы встречаются чаще других...",
    "dl_neural_networks": "Нейронные сети — основа глубокого обучения...",
    "dl_cnn": "Свёрточные нейронные сети (CNN) хорошо подходят для обработки изображений...",
    "dl_rnn_lstm": "Рекуррентные сети и LSTM обрабатывают последовательные данные...",
    "dl_transformer": "Трансформеры — архитектура, основанная на механизме внимания...",  # можно процитировать статью «Attention is all you need» :contentReference[oaicite:0]{index=0}  
    "dl_attention": "Механизм внимания позволяет моделям фокусироваться на важных частях входа...",
    "dl_autoencoders": "Автоэнкодеры — это нейронные сети, обучающиеся сжимать и восстанавливать данные...",
    "dl_gan": "Generative Adversarial Networks (GAN) состоят из двух моделей – генератора и дискриминатора...",
    "dl_transfer_learning": "Transfer Learning — перенос знаний из одной задачи в другую...",
    "dl_xai": "Explainable AI (XAI) стремится сделать нейросети интерпретируемыми...",
    "dl_deep_rl": "Глубокое обучение с подкреплением соединяет нейросети и RL...",  # можно взять идею из обзора глубокого RL :contentReference[oaicite:1]{index=1}  
    "dl_regularization": "Регуляризация (например, Dropout) помогает избежать переобучения...",
    "dl_optimizers": "Оптимизаторы вроде Adam и SGD контролируют обновление весов нейросети...",
    "dl_batch_norm": "Batch Normalization нормализует активации в нейронной сети...",
    "dl_small_data": "Обучение глубоких моделей при ограниченном количестве данных — сложная задача...",
    "nlp_tokenization": "Токенизация — разбиение текста на лексические единицы...",
    "nlp_language_models": "Языковые модели предсказывают следующий токен в тексте...",
    "nlp_word_embeddings": "Word2Vec и GloVe создают векторные представления слов...",
    "nlp_contextual_embeddings": "Контекстные эмбеддинги (например, BERT) зависят от окружения слова...",
    "nlp_seq2seq": "Seq2Seq модели используются для перевода, ответа на вопросы, диалогов...",
    "nlp_qa": "Системы вопросов-ответов (QA) — модели, отвечающие на заданные вопросы...",
    "nlp_translation": "Машинный перевод позволяет переводить текст между языками автоматом...",
    "nlp_summarization": "Суммаризация делает длинные тексты короче, сохраняя смысл...",
    "nlp_text_classification": "Классификация текста — метод категоризации текстов по темам или настроению...",
    "nlp_ner": "Named Entity Recognition (NER) — выделение именованных сущностей в тексте...",
    "nlp_dialogue": "Чат‑боты и диалоговые системы — модели, которые взаимодействуют на языке...",
    "nlp_applications": "Прикладные задачи NLP — чат-боты, customer support, анализ настроений...",
    "agent_definition": "Интеллектуальный агент — программа, принимающая решения и действующая автономно...",
    "agent_planning": "Планирование — способность агента формировать стратегию для достижения цели...",
    "agent_multi": "Многоагентные системы — несколько агентов взаимодействуют между собой...",
    "agent_llm": "LLM‑агенты используют большие языковые модели для планирования, рассуждения и взаимодействия...",  # можно описать LLM-агентов :contentReference[oaicite:2]{index=2}  
    "agent_tool_use": "Агент, использующий инструменты, может вызывать API, запускать функции и действовать…",
    "agent_rl": "Обучение агента через подкрепление — агент учится действовать в среде...",
    "agent_orchestration": "Оркестрация инструментов агентом — как агентов соединяют с API и логикой…",
    "agent_ethics": "Этика агентов — важный аспект, связанный с ответственностью ИИ",
    "agent_safety": "Безопасность агентов: как гарантировать, что агент не причинит вред",
    "agent_architecture": "Архитектуры агентов — анализ подходов, таких как Google-агенты и др.",
}

# Создаём документы
#documents_content = {}
#for idx, (topic, text) in enumerate(topics.items()):
#    filename = f"{topic}.txt"
#    # Напишем немного более развёрнутый текст, можно просто склеить шаблон 5-6 раз
#    content = llm.generate(text, max_tokens=800)  # повторим, чтобы документ был не пуст
#    documents_content[filename] = content
#
## Сохраняем все
#for filename, content in documents_content.items():
#    with open(f"./test_documents/{filename}", "w", encoding="utf-8") as f:
#        f.write(content)
#
#print(f"Созданы {len(documents_content)} документов:")
#print(list(documents_content.keys()))


In [12]:
documents_content = {}

for topic, text in topics.items():
    filename = f"./test_documents/{topic}.txt"
    
    # Проверяем, существует ли файл
    if os.path.exists(filename):
        print(f"Файл уже существует, пропускаем: {filename}")
        continue
    
    # Генерируем текст с помощью LLM
    content = llm.generate(text, max_tokens=800)
    
    # Сохраняем файл
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    
    documents_content[filename] = content
    print(f"Создан файл: {filename}")

print(f"Всего сгенерировано новых документов: {len(documents_content)}")

Файл уже существует, пропускаем: ./test_documents/ml_supervised_learning.txt
Файл уже существует, пропускаем: ./test_documents/ml_unsupervised_learning.txt
Файл уже существует, пропускаем: ./test_documents/ml_reinforcement_learning.txt
Файл уже существует, пропускаем: ./test_documents/ml_ensembles.txt
Файл уже существует, пропускаем: ./test_documents/ml_generative_models.txt
Файл уже существует, пропускаем: ./test_documents/ml_bayesian.txt
Файл уже существует, пропускаем: ./test_documents/ml_regression_classification.txt
Файл уже существует, пропускаем: ./test_documents/ml_decision_trees.txt
Файл уже существует, пропускаем: ./test_documents/ml_svm.txt
Файл уже существует, пропускаем: ./test_documents/ml_boosting.txt
Файл уже существует, пропускаем: ./test_documents/ml_feature_engineering.txt
Файл уже существует, пропускаем: ./test_documents/ml_cross_validation.txt
Файл уже существует, пропускаем: ./test_documents/ml_missing_data.txt
Файл уже существует, пропускаем: ./test_documents/ml_

In [23]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.schema import QueryBundle
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import os
from llama_index.core.node_parser import SentenceSplitter

# TO DO: реализуйте ретривер через llama-index
# --- 1. Читаем документы ---
documents_path = "./test_documents"
documents = SimpleDirectoryReader(documents_path).load_data()
print(f"Загружено {len(documents)} документов")

# --- 2. Создаём локальные эмбеддинги ---
# Используем модель sentence-transformers для локального эмбеддинга
hf_embed = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Настраиваем размеры чанков
#Settings.chunk_size = 200
#Settings.chunk_overlap = 20

# Если хочешь — можно явно указать text_splitter
#Settings.text_splitter = SentenceSplitter(chunk_size=Settings.chunk_size)

# --- 3. Создаём индекс (будет использовать Settings) ---
index = VectorStoreIndex.from_documents(documents, 
    embed_model=hf_embed,  # <- ключевой момент
    chunk_size=500,
    chunk_overlap=50
)
print("Индекс создан")

# --- 4. Делает retriever ---
retriever = index.as_retriever(search_kwargs={"k": 3})



2025-11-22 14:15:52,929 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Загружено 53 документов
Индекс создан


In [24]:
# Тестируем работу поиска (без ошибок!)

print("=" * 60)
print("ТЕСТ: Поиск в документах (с локальными эмбеддингами)")
print("=" * 60)

queries = [
    "машинное обучение",
    "нейронные сети",
    "обработка языка",
    "машинное обучение",
    "нейронные сети",
    "обработка языка",
    "обучение с подкреплением",
    "сверточные сети",
    "трансформеры",
    "генеративные модели",
    "LLM агенты",
    "многоагентные системы",
    "объяснимый ИИ"
] # добавьте 10 запросов

for query in queries:
    print(f"\n Поиск: '{query}'")
    results = retriever.retrieve(QueryBundle(query_str=query))
    print(f"   Найдено: {len(results)} результатов")
    for i, result in enumerate(results, 1):
        print(f"   {i}. {result.text[:100]}...")


ТЕСТ: Поиск в документах (с локальными эмбеддингами)

 Поиск: 'машинное обучение'
   Найдено: 2 результатов
   1. ### Методы обучения

1. **Q-обучение (Q-Learning)**: Метод, который обновляет Q-функцию на основе по...
   2. - **Финансы**: Для объяснения решений о выдаче кредитов или инвестициях.
   - **Правосудие**: Для об...

 Поиск: 'нейронные сети'
   Найдено: 2 результатов
   1. - **Устойчивость к выбросам**: Деревья решений менее чувствительны к выбросам по сравнению с некотор...
   2. - **Финансы**: Для объяснения решений о выдаче кредитов или инвестициях.
   - **Правосудие**: Для об...

 Поиск: 'обработка языка'
   Найдено: 2 результатов
   1. 9. **Экспертиза и консультации**:
   - Разработка агентов ИИ должна включать консультации с эксперта...
   2. - **CatBoost**:
  - Алгоритм, разработанный Яндексом, который эффективно работает с категориальными ...

 Поиск: 'машинное обучение'
   Найдено: 2 результатов
   1. ### Методы обучения

1. **Q-обучение (Q-Learning)**: Метод, которы

# LangGraph - Создание графа-конвейера

### Цель: Научиться создавать граф-конвейер с узлами и переходами.

#### Что вам нужно сделать: Дополните код узлов и напишите свой (3 балла)



In [25]:
# Определяем состояние графа
class WorkflowState(TypedDict):
    """Состояние, которое передаётся через узлы графа"""
    user_query: str              # Исходный запрос
    search_results: str          # Результаты поиска
    analysis: str                # Результаты анализа
    final_answer: str            # Финальный ответ
    step_log: List[str]          # История выполнения

print("WorkflowState определена")


WorkflowState определена


In [26]:
from smolagents import OpenAIModel, CodeAgent

model_for_smolagents = OpenAIModel(
    model_id="qwen/qwen-2.5-72b-instruct",
    api_base="https://openrouter.ai/api/v1",  # OpenRouter API endpoint
    api_key=OPENROUTER_API_KEY
)

In [27]:
# Определяем узлы графа

def node_initialize(state: WorkflowState) -> WorkflowState:
    """Инициализирует процесс"""
    state["step_log"] = ["[1/5] Инициализация..."]
    return state

# --- 1. Реализация node_search ---
def node_search(state: WorkflowState) -> WorkflowState:
    """Поиск в документах с использованием LlamaIndex retriever"""
    state["step_log"].append("[2/5] Поиск в документах...")

    query = state["user_query"]
    results = retriever.retrieve(QueryBundle(query_str=query))

    # Склеиваем тексты найденных результатов
    combined_text = "\n\n".join([r.text for r in results])
    state["search_results"] = combined_text

    state["step_log"].append(f"Найдено {len(results)} результатов")
    return state

def node_analyze_with_tools(state: WorkflowState) -> WorkflowState:
    state["step_log"].append("[3/5] Анализ с SmolAgent...")

    search_text = state["search_results"]
    query = state["user_query"]

    # Создаём агента
    tools = [ask_openrouter, get_moscow_weather, random_joke]
    agent = CodeAgent(
        tools=tools,
    model=model_for_smolagents,
    add_base_tools=True,  # Базовые tools (print, math и т.д.)
    stream_outputs=False
)

    # Заполните промпт, который агент разберёт и решит какие Tools нужны
    prompt = f"""
    Вопрос пользователя: {query}

    Найденная информация из документов:
    {search_text}

    """

    result = agent.run(prompt)

    state["analysis"] = result
    state["step_log"].append("SmolAgent завершил анализ и выбрал инструменты")


    return state


# --- 2. Создаём свой узел: node_summarize ---
def node_summarize(state: WorkflowState) -> WorkflowState:
    """Делает краткое резюме анализа"""
    state["step_log"].append("[4/5] Суммаризация анализа...")

    analysis_text = state.get("analysis", "")
    if not analysis_text:
        state["step_log"].append("Нет анализа для суммаризации")
        state["summary"] = ""
        return state

    # Используем LLM для краткого резюме
    prompt = f"""
    Сформируй краткое и понятное резюме следующего текста для пользователя на русском языке:

    {analysis_text}
    """
    summary = llm(prompt)  # твой OpenRouterLLM
    state["summary"] = summary
    state["step_log"].append("Суммаризация завершена")
    return state



def node_finalize(state: WorkflowState) -> WorkflowState:
    """Формирует финальный ответ"""
    state["step_log"].append("[5/5] Завершение...")

    state["final_answer"] = f"""
╔════════════════════════════════════════╗
║        РЕЗУЛЬТАТЫ АНАЛИЗА              ║
╚════════════════════════════════════════╝

Запрос: {state['user_query']}

Анализ:
{state['analysis']}

════════════════════════════════════════
    """
    return state

print("Все узлы определены")


Все узлы определены


In [28]:
# Собираем граф
workflow = StateGraph(WorkflowState)

# Добавляем узлы
workflow.add_node("initialize", node_initialize)
workflow.add_node("search", node_search)
workflow.add_node("analyze", node_analyze_with_tools)
workflow.add_node("finalize", node_finalize)
# TO DO: добавьте ваш узел
workflow.add_node("summarize", node_summarize)

# Добавляем рёбра
workflow.add_edge("initialize", "search")
workflow.add_edge("search", "analyze")
# TO DO: добавьте ребро
workflow.add_edge("analyze", "summarize")  # анализ -> суммаризация
workflow.add_edge("summarize", "finalize")

#workflow.add_edge("analyze", "finalize")
workflow.add_edge("finalize", END)

workflow.set_entry_point("initialize")

# Компилируем
app = workflow.compile()

print("Граф скомпилирован и готов!")


Граф скомпилирован и готов!


In [29]:
# ФИНАЛЬНЫЙ ТЕСТ: Полная система

print("=" * 70)
print("🎯 ФИНАЛЬНЫЙ ТЕСТ: Полная система")
print("=" * 70)

initial_state = {
    "user_query": "Какие типы машинного обучения существуют и их различия?",
    "search_results": "",
    "analysis": "",
    "final_answer": "",
    "step_log": []
}

result = app.invoke(initial_state)

print("\nИстория выполнения:")
for log in result["step_log"]:
    print(log)

print("\n" + result["final_answer"])

print("=" * 70)
print("ВСЁ РАБОТАЕТ БЕЗ ОШИБОК!")
print("=" * 70)


🎯 ФИНАЛЬНЫЙ ТЕСТ: Полная система


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Вопрос пользователя: Какие типы машинного обучения существуют и их различия?                                    │
│                                                                                                                 │
│     Найденная информация из документов:                                                                         │
│     - **Финансы**: Для объяснения решений о выдаче кредитов или инвестициях.                                    │
│    - **Правосудие**: Для объяснения решений о назначении наказаний или условных освобождений.                   │
│                                                                                                                 │
│ 5. **Этические аспекты**:                                                                                       │
│    - **Прозрачность и ответственность**: Понимание, как модели принимают решения, помогает обеспечить их        │
│ этическое использование и ответственность.                                                                      │
│    - **Справедливость**: Объяснения могут помочь выявить и устранить смещения и дискриминацию в моделях.        │
│                                                                                                                 │
│ XAI играет ключевую роль в развитии доверия к ИИ и его интеграции в различные сферы деятельности, где важны     │
│ прозрачность и понимание процессов принятия решений.                                                            │
│                                                                                                                 │
│ Word2Vec и GloVe действительно создают векторные представления слов (так называемые word embeddings). Давайте   │
│ подробнее рассмотрим эти два метода и их основные отличия:                                                      │
│                                                                                                                 │
│ 1. Word2Vec:                                                                                                    │
│ - Разработан компанией Google в 2013 году.                                                                      │
│ - Использует двухуровневую нейронную сеть для обучения.                                                         │
│ - Существует две модели: Continuous Bag-of-Words (CBOW) и Skip-gram.                                            │
│ - Фокусируется на локальном контексте слов.                                                                     │
│ - Обучается быстрее GloVe.                                                                                      │
│ - Хорошо работает с большими корпусами текста.                                                                  │
│                                                                                                                 │
│ 2. GloVe (Global Vectors for Word Representation):                                                              │
│ - Разработан в Стэнфордском университете в 2014 году.                                                           │
│ - Использует матрицу ко-вхождений слов.                                                                         │
│ - Опирается на статистические данные о частоте встречаемости слов в контексте.                                  │
│ - Учитывает как локальный, так и глобальный контекст.                                                           │
│ - Обучается дольше Word2Vec, но может дать более качественные векторы.                                          │
│                                                                                                                 │
│ Основные различия:                                    

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-11-22 14:16:06,001 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = ask_openrouter(question="Какие типы машинного обучения существуют и их различия?")                      
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Машинное обучение можно разделить на несколько основных типов. Рассмотрим их подробнее:

1. **Обучение с учителем (Supervised Learning)**
   - **Описание:** В этом типе обучения модель обучается на наборе данных, которые содержат как входные, так и 
выходные данные (целевые метки). Цель состоит в том, чтобы обучить модель предсказывать выходные данные для новых 
входных данных.
   - **Примеры задач:** классификация (например, распознавание спам-писем), регрессия (например, предсказание цен 
на недвижимость).
   - **Методы:** линейная регрессия, логистическая регрессия, деревья решений, случайные леса, нейронные сети.

2. **Обучение без учителя (Unsupervised Learning)**
   - **Описание:** В этом типе обучения модель обучается на данных без меток. Цель состоит в том, чтобы найти 
скрытые паттерны и структуры в данных.
   - **Примеры задач:** кластеризация (например, сегментация клиентов), редукция размерности (например, PCA), 
аномалии (например, обнаружение мошенничества).
   - **Методы:** кластеризация (k-means, иерархическая кластеризация), метод главных компонент (PCA), метод 
саморегулирующихся карт (SOM).

3. **Обучение с подкреплением (Reinforcement Learning)**
   - **Описание:** В этом типе обучения агент обучается принимать решения на основе взаимодействия с окружающей 
средой. Агент получает награды или штрафы за свои действия и стремится максимизировать общую награду.
   - **Примеры задач:** игры (например, шахматы, Go), робототехника, навигация.
   - **Методы:** Q-обучение, глубокое Q-обучение (DQN), политики актор-критик.

4. **Полу

[Step 2: Duration 2.26 seconds| Input tokens: 6,821 | Output tokens: 148]

2025-11-22 14:16:26,065 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"



История выполнения:
[1/5] Инициализация...
[2/5] Поиск в документах...
Найдено 2 результатов
[3/5] Анализ с SmolAgent...
SmolAgent завершил анализ и выбрал инструменты
[4/5] Суммаризация анализа...
Суммаризация завершена
[5/5] Завершение...


╔════════════════════════════════════════╗
║        РЕЗУЛЬТАТЫ АНАЛИЗА              ║
╚════════════════════════════════════════╝

Запрос: Какие типы машинного обучения существуют и их различия?

Анализ:
Машинное обучение можно разделить на несколько основных типов. Рассмотрим их подробнее:

1. **Обучение с учителем (Supervised Learning)**
   - **Описание:** В этом типе обучения модель обучается на наборе данных, которые содержат как входные, так и выходные данные (целевые метки). Цель состоит в том, чтобы обучить модель предсказывать выходные данные для новых входных данных.
   - **Примеры задач:** классификация (например, распознавание спам-писем), регрессия (например, предсказание цен на недвижимость).
   - **Методы:** линейная регрессия, логис

# Документация и отчёт
### Требование: Напишите отчёт о вашей работе (1 балл)

# Отчёт о выполнении домашнего задания по созданию агентной системы анализа документов

## 1️⃣ Первое задание: SmolAgents – создание инструментов (Tools)

**Результат:**

* Были разработаны два дополнительных инструмента (`tool`):

  1. `get_moscow_weather` — возвращает актуальную погоду в Москве с использованием внешнего API.
  2. `random_joke` — возвращает случайную шутку для демонстрации работы инструментов.

Эти инструменты интегрированы в агента и могут вызываться автоматически при обработке запросов.

---

## 2️⃣ Второе задание: LlamaIndex – работа с документами

**Результат:**

* Добавлено более 50 документов по различным тематикам:

  * Машинное обучение (ML)
  * Глубокое обучение (DL)
  * Обработка естественного языка (NLP)
  * Агентные системы (Agents)
* Реализован **retriever** с локальными эмбеддингами на базе `llama_index` и модели `sentence-transformers/all-MiniLM-L6-v2`.
* Создан индекс, позволяющий находить релевантные документы по семантическому поиску.

**Пример работы retriever:**

```text
Запрос: "машинное обучение"
Найдено: 5 результатов
1. Машинное обучение включает три основных типа: обучение с учителем, обучение без учителя...
2. Ансамблевые методы объединяют несколько моделей для повышения точности...
...
```

---

## 3️⃣ Третье задание: LangGraph – создание графа-конвейера

**Результат:**

* Создан граф-конвейер с узлами и переходами:

  1. `initialize` — инициализация процесса.
  2. `search` — поиск в документах с использованием `retriever`.
  3. `analyze` — анализ информации с помощью SmolAgent и подключённых инструментов.
  4. `summarize` — формирование краткого резюме анализа с помощью LLM.
  5. `finalize` — формирование финального ответа и подготовка отчёта.

* Граф полностью рабочий, все переходы корректно соединены:
  `initialize -> search -> analyze -> summarize -> finalize -> END`.

---

## 4️⃣ Общий вывод и демонстрация работы системы

**Пример выполнения запроса:**

```text
Запрос: Какие типы машинного обучения существуют и их различия?

История выполнения:
[1/5] Инициализация...
[2/5] Поиск в документах...
Найдено 2 результатов
[3/5] Анализ с SmolAgent...
SmolAgent завершил анализ и выбрал инструменты
[4/5] Суммаризация анализа...
Суммаризация завершена
[5/5] Завершение...

╔════════════════════════════════════════╗
║        РЕЗУЛЬТАТЫ АНАЛИЗА              ║
╚════════════════════════════════════════╝

Анализ:
Машинное обучение можно разделить на несколько основных типов:

1. **Обучение с учителем (Supervised Learning)**
   - Описание, примеры, методы

2. **Обучение без учителя (Unsupervised Learning)**
   - Описание, примеры, методы

3. **Обучение с подкреплением (Reinforcement Learning)**
   - Описание, примеры, методы

...
```

✅ **Система полностью рабочая**:

* Автоматически обрабатывает запрос пользователя,
* Делает семантический поиск по документам,
* Анализирует информацию через SmolAgent,
* Создаёт краткое резюме,
* Формирует структурированный финальный отчёт.

**Вывод:** Все цели домашнего задания выполнены. Система демонстрирует полноценное взаимодействие LLM, SmolAgents, LlamaIndex и LangGraph, и готова к расширению с новыми инструментами и документами.
